In [1]:
!pip3 install beautifulsoup4
!pip install ete3

DEPRECATION: Loading egg at /Users/phr361/anaconda3/lib/python3.11/site-packages/pyrosetta-2022.25+release.c55c9eac6e7-py3.11-macosx-11.0-arm64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /Users/phr361/anaconda3/lib/python3.11/site-packages/pyrosetta-2022.25+release.c55c9eac6e7-py3.11-macosx-11.0-arm64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [2]:
# import libraries
import urllib.request
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
# get download link for all refseq from padloc server
def get_link(page_num):
    quote_page = 'https://padloc.otago.ac.nz/padloc/refseq/?page=' + str(page_num)
    page = urllib.request.urlopen(quote_page)
    soup = BeautifulSoup(page, 'html.parser')
    item_list = soup.find_all("td")
    genome_id_list, genome_name_list, link_list = [], [], []
    for i in range(0, len(item_list), 3):
        genome_id_list.append(item_list[i].text)
        genome_name_list.append(item_list[i+1].text)
        if item_list[i+2].text == "No System Detected":
            link_list.append(item_list[i+2].text)
        else:
            link_list.append(item_list[i+2].find("a")["href"])
    return genome_id_list, genome_name_list, link_list

import os
import urllib.request
import urllib.error
from IPython.display import clear_output
from tqdm import tqdm

def download_file(ref_id, ref_link):
    num_id = ref_link.strip().split("/")[-1]
    download_link = f"https://padloc.otago.ac.nz/static/results/csv/{num_id}_padloc.csv"
    file_name = f"{ref_id}_padloc.csv"
    
    if os.path.exists(file_name):
        print(f"File '{file_name}' already exists. Skipping download.")
        return



    print(f"Attempting to download: {download_link}")

    try:
        urllib.request.urlretrieve(download_link, file_name)
        print(f"Downloaded '{file_name}'.")
    except urllib.error.HTTPError as e:
        print(f"HTTP Error {e.code}: {e.reason} for URL {download_link}")
    except urllib.error.URLError as e:
        print(f"URL Error: {e.reason} for URL {download_link}")
    # Clear output but keep the progress bar
    clear_output(wait=True)


'def download_file(ref_id, ref_link):\n    num_id = ref_link.strip().split("/")[-1]\n    download_link = f"https://padloc.otago.ac.nz/static/results/csv/{num_id}_padloc.csv"\n    file_name = f"{ref_id}_padloc.csv"\n    \n    if os.path.exists(file_name):\n        print(f"File \'{file_name}\' already exists. Skipping download.")\n        return\n\n    print(f"Attempting to download: {download_link}")\n\n    try:\n        urllib.request.urlretrieve(download_link, file_name)\n        print(f"Downloaded \'{file_name}\'.")\n    except urllib.error.HTTPError as e:\n        print(f"HTTP Error {e.code}: {e.reason} for URL {download_link}")\n    except urllib.error.URLError as e:\n        print(f"URL Error: {e.reason} for URL {download_link}")\n\ndef download_file(ref_id, ref_link):\n    num_id = ref_link.strip().split("/")[-1]\n    \n    \n    file_name = f"{ref_id}_padloc.csv"\n    \n    if os.path.exists(file_name):\n        print(f"File \'{file_name}\' already exists. Skipping download.")\n

In [4]:
from joblib import Parallel, delayed
import time
start = time.time()
# n_jobs is the number of parallel jobs
res = Parallel(n_jobs=20)(delayed(get_link)(i) for i in range(1, 2342))
end = time.time()
print('{:.4f} s'.format(end-start))

421.6473 s


In [5]:
genome_id_list, genome_name_list, link_list = [], [], []
for item in res:
    genome_id_list.extend(item[0])
    genome_name_list.extend(item[1])
    link_list.extend(item[2])

padoc_refseq_data = pd.DataFrame({"Genome_id": genome_id_list, 
                                  "Genome_name": genome_name_list,
                                  "Link": link_list})

filter_padoc_refseq_data = padoc_refseq_data.loc[padoc_refseq_data['Link'] != "No System Detected"].reset_index().drop_duplicates("Link")

In [6]:
filter_padoc_refseq_data = filter_padoc_refseq_data.reset_index(drop=True)

In [7]:
filter_padoc_refseq_data

,index,Genome_id,Genome_name,Link
0,0,GCF_002973605.1,Abditibacterium utsteinense LMG 29911,/padloc/refseq/68af4c71e8da7ff5348a5fdb07d2259...
1,1,GCF_000160075.2,Abiotrophia defectiva ATCC 49176,/padloc/refseq/d436425611f2c8c6617b0c0a29160cc...
2,2,GCF_013267415.1,Abiotrophia defectiva FDAARGOS_785,/padloc/refseq/f08fa30a7eaebc01d96b6d7cccee5f7...
3,3,GCF_001815865.1,Abiotrophia sp. HMSC24B09,/padloc/refseq/b5b8aef2bb6aae6e35fd5ad5635ae41...
4,4,GCF_003725415.1,Absicoccus porci YH-panp20,/padloc/refseq/a940472b2015b4a4489c31d8edd04f3...
...,...,...,...,...
223247,234093,GCF_000218875.1,Zymomonas mobilis subsp. pomaceae ATCC 29192,/padloc/refseq/d82e3f8efd56652d6382eb1c47c009c...
223248,234094,GCF_006539385.1,Zymomonas mobilis subsp. pomaceae NBRC 13757,/padloc/refseq/6251c466ce5a0c9ac210ce697b12777...
223249,234095,GCF_004168345.2,Zymomonas mobilis ZM4*,/padloc/refseq/0ee1435c7baa7a72e33b481c10c9e9f...
223250,234096,GCF_019316745.1,Zymomonas mobilis ZM401,/padloc/refseq/907bd97599da364d7d17f6eac3a5fb2...


In [9]:
import time
from tqdm import tqdm
import concurrent.futures

start = time.time()
num_threads = 8
def download_wrapper(row):
    download_file(row.Genome_id, row.Link)

with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
    list(tqdm(executor.map(download_wrapper, filter_padoc_refseq_data.itertuples(index=False)), 
              total=len(filter_padoc_refseq_data), desc="Downloading", unit="file"))

end = time.time()
print(f"Elapsed time: {end - start:.4f} s")

filter_padoc_refseq_data.to_csv("filter_data.csv", index=None, header=None)

Downloading: 100%|█████████████████| 223252/223252 [00:09<00:00, 23056.59file/s]

Elapsed time: 11.8233 s


In [12]:
import dask.dataframe as dd

In [13]:
!pwd

/Users/phr361/Documents/Coding/ProteinDesign/AntiAntiDefence/Shango_v2/PhyloGeneticTree/Padloc


In [22]:
import pandas as pd
import glob
import os
from tqdm import tqdm  # Import tqdm for progress bar

files = glob.glob("./*.csv")
df_list = []

# Wrap the file list with tqdm for progress tracking
for file in tqdm(files, desc="Processing files"):
    try:
        chunk = pd.read_csv(file, low_memory=False)
        if chunk.empty:  # Check if the file is empty
            print(f"Skipping empty file: {file}")
            continue
        chunk["Assembly"] = os.path.basename(file)  # Extract filename only
        df_list.append(chunk)
    except pd.errors.EmptyDataError:
        print(f"Skipping empty or corrupted file: {file}")
    except pd.errors.ParserError:
        print(f"Skipping malformed CSV file: {file}")

# Concatenate all dataframes into one
if df_list:  # Ensure there is data to concatenate
    df = pd.concat(df_list, ignore_index=True)
else:
    print("No valid CSV files found.")
del df_list

Processing files:   3%|▍                 | 5585/223343 [00:38<20:42, 175.23it/s]

Skipping empty or corrupted file: ./GCF_008042335.1_padloc.csv


Processing files:  11%|█▊               | 23477/223343 [03:22<32:34, 102.28it/s]

Skipping malformed CSV file: ./GCF_000227945.1_padloc.csv


Processing files:  17%|███               | 38285/223343 [05:55<32:11, 95.80it/s]

...

Processing files:  77%|████████████▎   | 172203/223343 [26:05<08:24, 101.35it/s]

Skipping malformed CSV file: ./GCF_001647185.1_padloc.csv


Processing files:  99%|████████████████▋| 220017/223343 [36:48<01:12, 46.00it/s]

Skipping malformed CSV file: ./GCF_003336925.1_padloc.csv


Processing files: 100%|█████████████████| 223343/223343 [37:35<00:00, 99.02it/s]


In [23]:
import pandas as pd
import numpy as np

def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reduce the memory usage of a DataFrame by converting data types.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        
    Returns:
        pd.DataFrame: The optimized DataFrame with reduced memory usage.
    """
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:  # Exclude object columns
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            elif str(col_type)[:5] == 'float':
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype('category')
    
    return df


In [24]:
df = reduce_memory_usage(df)

/var/folders/k0/7lzlp98x6c188pkk0rzbn2d40000gn/T/ipykernel_34815/2993679905.py:31: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/var/folders/k0/7lzlp98x6c188pkk0rzbn2d40000gn/T/ipykernel_34815/2993679905.py:31: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/var/folders/k0/7lzlp98x6c188pkk0rzbn2d40000gn/T/ipykernel_34815/2993679905.py:31: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:


In [25]:
df

/Users/phr361/anaconda3/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/Users/phr361/anaconda3/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,system.number,seqid,system,target.name,hmm.accession,hmm.name,protein.name,full.seq.E.value,domain.iE.value,target.coverage,...,target.description,relative.position,contig.end,all.domains,best.hits,Assembly,0,GCF_002973605.1,Abditibacterium utsteinense LMG 29911,/padloc/refseq/68af4c71e8da7ff5348a5fdb07d2259ef6547336f5e76b7f35b56f8df39ecb5e
0,1.0,NZ_PQJK01000001.1,GAO_19,WP_113525832.1,PD-T7-2_A_WP_176695388.1,PD-T7-2_A_WP_176695388.1,SIR2,0.0,0.0,0.984863,...,SIR2 family protein [Acidithiobacillus ferridu...,617.0,657.0,"PD-T7-2_A_WP_024466888.1,PD-T7-2_A_WP_02446688...","PD-T7-2_A_WP_176695388.1, PD-T7-2_A_WP_1766953...",GCF_003309025.1_padloc.csv,NaN,NaN,NaN,NaN
1,1.0,NZ_PQJK01000001.1,GAO_19,WP_113525833.1,PD-T7-2_B_WP_212464868.1,PD-T7-2_B_WP_212464868.1,HerA,0.0,0.0,0.965820,...,ATP-binding protein [Acidithiobacillus ferridu...,618.0,657.0,"PD-T7-2_B_WP_009660643.1,PD-T7-2_B_WP_00966064...","PD-T7-2_B_WP_212464868.1, PD-T7-2_B_WP_2124648...",GCF_003309025.1_padloc.csv,NaN,NaN,NaN,NaN
2,1.0,NZ_PQJK01000001.1,RM_type_I,pseudo_subWP_012536387.1,PDLC03040,REase_I_00001,REase_I,0.0,0.0,0.863770,...,HsdR family type I site-specific deoxyribonucl...,353.0,657.0,"PDLC03040,REase_I_00001, 1, 4.7e-91, 0.864, 0....","PDLC03040, REase_I_00001, 4.7e-91, 0.864, 0.87...",GCF_003309025.1_padloc.csv,NaN,NaN,NaN,NaN
3,1.0,NZ_PQJK01000001.1,RM_type_I,WP_113525657.1,PDLC03134,Specificity_I_00083,Specificity_I,0.0,0.0,0.964844,...,restriction endonuclease subunit S [Acidithiob...,356.0,657.0,"PDLC03019,MTase_I_00002, 1, 2.1e-05, 0.572, 0....","PDLC03134, Specificity_I_00083, 4.2e-81, 0.965...",GCF_003309025.1_padloc.csv,NaN,NaN,NaN,NaN
4,1.0,NZ_PQJK01000001.1,RM_type_I,WP_113525659.1,PDLC03031,MTase_I_00014,MTase_I,0.0,0.0,0.910156,...,type I restriction-modification system subunit...,359.0,657.0,"PDLC03018,MTase_I_00001, 1, 8.1e-07, 0.132, 0....","PDLC03028, MTase_I_00011, 5.4e-163, 0.985, 0.4...",GCF_003309025.1_padloc.csv,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6551076,8.0,NZ_MZXM01000011.1,PDC-S07,WP_000736053.1,PDLC04899,PDC-S07_WP_110056698.1,PDC-S07,0.0,0.0,0.969238,...,8-oxo-dGTP diphosphatase MutT [Salmonella],116.0,198.0,"PDLC04899,PDC-S07_WP_110056698.1, 1, 2.5e-41, ...","PDLC04899, PDC-S07_WP_110056698.1, 2.5e-41, 0....",GCF_002042745.1_padloc.csv,NaN,NaN,NaN,NaN
6551077,9.0,NZ_MZXM01000012.1,PDC-S05,WP_000774140.1,PDLC04852,PDC-S05_WP_110807286.1,PDC-S05,0.0,0.0,0.974121,...,virulence RhuM family protein [Salmonella],55.0,118.0,"PDLC04843,PDC-S05_WP_181573034.1, 1, 1.6e-110,...","PDLC04852, PDC-S05_WP_110807286.1, 2e-171, 0.9...",GCF_002042745.1_padloc.csv,NaN,NaN,NaN,NaN
6551078,10.0,NZ_MZXM01000013.1,PDC-S04,WP_001280665.1,PDLC04838,PDC-S04_WP_169250289.1,PDC-S04,0.0,0.0,0.979980,...,putative adenosine monophosphate-protein trans...,221.0,246.0,"PDLC04838,PDC-S04_WP_169250289.1, 1, 1.4e-74, ...","PDLC04838, PDC-S04_WP_169250289.1, 1.4e-74, 0....",GCF_002042745.1_padloc.csv,NaN,NaN,NaN,NaN
6551079,1.0,NZ_MZXM01000001.1,cbass_type_I,WP_000369761.1,PDLC00696,Effector_00034,Effector,0.0,0.0,0.983887,...,SLATT domain-containing protein [Salmonella],739.0,755.0,"PDLC00690,Effector_00028, 1, 4.8e-19, 0.956, 0...","PDLC00696, Effector_00034, 2.4e-81, 0.984, 0.9...",GCF_002042745.1_padloc.csv,NaN,NaN,NaN,NaN


In [26]:
df.to_csv("all_padloc.csv")